# 03 - Modeling: LogReg vs RandomForest (group-aware CV)

## Obiettivi didattici

1. Costruire la pipeline `feature_engineer -> preprocessor -> model`.
2. Implementare lo split train/test **group-aware** su `patient_nbr`.
3. Eseguire **StratifiedGroupKFold** con scoring `average_precision` (AUC-PR), adatto a classi sbilanciate.
4. Confrontare un baseline interpretabile (LogReg) e un ensemble (RandomForest), entrambi con `class_weight='balanced'`.


In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

from readmit_pipeline.pipeline import prepare_data, build_candidate_pipelines
from readmit_pipeline.features import ReadmissionFeatureEngineer
from readmit_pipeline.tuning import tune_all_models, summarize_tuning
from readmit_pipeline.config import PipelineConfig


## Split group-aware + costruzione pipeline


In [ ]:
config = PipelineConfig(cv_folds=3, random_state=42)
X_train, X_test, y_train, y_test, g_train, g_test, df_clean = prepare_data(config)
fe = ReadmissionFeatureEngineer()
X_train_fe = fe.fit_transform(X_train)
pipelines = build_candidate_pipelines(X_train_fe)
list(pipelines.keys())


## Tuning (GridSearchCV + StratifiedGroupKFold)


In [ ]:
results = tune_all_models(
    pipelines=pipelines,
    X=X_train, y=y_train, groups=g_train,
    config=config,
)
summarize_tuning(results)
